In [6]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [7]:
import os

folder_path = '/content/drive/MyDrive/COMVIS/CITRA APEL'

os.chdir(folder_path)

for file_name in os.listdir(folder_path):
    print(f"Nama File: {file_name}")

Nama File: Apel 1.jpg
Nama File: Apel 7.jpg
Nama File: Apel 8.jpg
Nama File: Apel 9.jpg
Nama File: Apel 10.jpg
Nama File: Apel 2.jpg
Nama File: Apel 3.jpg
Nama File: Apel 4.jpg
Nama File: Apel 5.jpg
Nama File: Apel 6.jpg


In [8]:
import cv2
import os
import numpy as np
import pandas as pd

# Path dataset
folder_path = '/content/drive/MyDrive/COMVIS/CITRA APEL'

features = []
image_names = []
labels = []

for filename in os.listdir(folder_path):
    img_path = os.path.join(folder_path, filename)

    img = cv2.imread(img_path)

    if img is None:
        print(f"Gagal membaca: {filename}")
        continue

    # Konversi ke RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # =========================
    # 🔥 MASKING (BUANG BACKGROUND TERANG)
    # =========================
    # Ambil pixel yang bukan putih/terang
    mask = np.logical_and(
        img_rgb[:, :, 0] < 240,
        img_rgb[:, :, 1] < 240
    )

    # Ambil hanya area apel
    r = img_rgb[:, :, 0][mask]
    g = img_rgb[:, :, 1][mask]
    b = img_rgb[:, :, 2][mask]

    # Cegah error jika mask kosong
    if len(r) == 0:
        print(f"Mask gagal pada: {filename}")
        continue

    # Hitung mean warna
    mean_r = np.mean(r)
    mean_g = np.mean(g)
    mean_b = np.mean(b)

    # =========================
    # 🔥 LABEL DENGAN THRESHOLD
    # =========================
    if (mean_r - mean_g) > 20:
        label = "merah"
    elif (mean_g - mean_r) > 20:
        label = "hijau"
    else:
        label = "kuning"

    # =========================
    # 🔥 DEBUG PRINT
    # =========================
    print(f"{filename} -> R:{mean_r:.2f}, G:{mean_g:.2f}, B:{mean_b:.2f} => {label}")

    # Simpan data
    features.append([mean_r, mean_g, mean_b])
    image_names.append(filename)
    labels.append(label)

# =========================
# BUAT DATAFRAME
# =========================
df = pd.DataFrame(features, columns=['R_mean', 'G_mean', 'B_mean'])
df['Nama_File'] = image_names
df['Label'] = labels

print("\n📊 HASIL EKSTRAKSI:")
print(df)

Apel 1.jpg -> R:131.52, G:59.79, B:41.15 => merah
Apel 7.jpg -> R:150.20, G:99.80, B:74.30 => merah
Apel 8.jpg -> R:150.16, G:99.58, B:74.19 => merah
Apel 9.jpg -> R:149.96, G:99.24, B:73.96 => merah
Apel 10.jpg -> R:149.81, G:98.68, B:73.62 => merah
Apel 2.jpg -> R:98.68, G:92.97, B:47.80 => kuning
Apel 3.jpg -> R:100.81, G:95.32, B:49.05 => kuning
Apel 4.jpg -> R:102.84, G:97.57, B:50.12 => kuning
Apel 5.jpg -> R:138.43, G:126.92, B:73.83 => kuning
Apel 6.jpg -> R:137.66, G:125.56, B:72.54 => kuning

📊 HASIL EKSTRAKSI:
       R_mean      G_mean     B_mean    Nama_File   Label
0  131.520810   59.793307  41.153543   Apel 1.jpg   merah
1  150.199144   99.803531  74.301458   Apel 7.jpg   merah
2  150.162618   99.581574  74.189168   Apel 8.jpg   merah
3  149.955500   99.240089  73.957600   Apel 9.jpg   merah
4  149.809150   98.681830  73.623007  Apel 10.jpg   merah
5   98.675641   92.972176  47.798660   Apel 2.jpg  kuning
6  100.811258   95.318118  49.051802   Apel 3.jpg  kuning
7  102.83

In [9]:
import pandas as pd

# Buat DataFrame
df = pd.DataFrame(features, columns=['R_mean', 'G_mean', 'B_mean'])

# Tambahkan kolom nama file & label
df['Nama_File'] = image_names
df['Label'] = labels

# Simpan ke CSV
csv_path = '/content/drive/MyDrive/COMVIS/CITRA APEL/fitur_apel.csv'
df.to_csv(csv_path, index=False)

print("Data berhasil disimpan ke CSV")
print(df.head())

Data berhasil disimpan ke CSV
       R_mean     G_mean     B_mean    Nama_File  Label
0  131.520810  59.793307  41.153543   Apel 1.jpg  merah
1  150.199144  99.803531  74.301458   Apel 7.jpg  merah
2  150.162618  99.581574  74.189168   Apel 8.jpg  merah
3  149.955500  99.240089  73.957600   Apel 9.jpg  merah
4  149.809150  98.681830  73.623007  Apel 10.jpg  merah


In [10]:
import pandas as pd
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load data dari CSV
data = pd.read_csv('/content/drive/MyDrive/COMVIS/CITRA APEL/fitur_apel.csv')

# Ambil fitur & label
X = data[['R_mean', 'G_mean', 'B_mean']]
y = data['Label']

# Split data (train & test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model Naive Bayes
model = GaussianNB()
model.fit(X_train, y_train)

# Prediksi data test
y_pred = model.predict(X_test)

# Hitung akurasi
akurasi = accuracy_score(y_test, y_pred)

# =========================
# TAMPILKAN HASIL
# =========================

print("HASIL KLASIFIKASI\n")

print("Akurasi Model:", akurasi)
print("\nData Uji:")
print(X_test)

print("\nLabel Asli:")
print(y_test.values)

print("\nHasil Prediksi:")
print(y_pred)

# Tambahkan hasil ke DataFrame
data['Prediksi'] = model.predict(X)

print("\nDATA LENGKAP DENGAN PREDIKSI:")
print(data)

HASIL KLASIFIKASI

Akurasi Model: 0.5

Data Uji:
       R_mean      G_mean     B_mean
8  138.429812  126.915051  73.832562
1  150.199144   99.803531  74.301458

Label Asli:
['kuning' 'merah']

Hasil Prediksi:
['merah' 'merah']

DATA LENGKAP DENGAN PREDIKSI:
       R_mean      G_mean     B_mean    Nama_File   Label Prediksi
0  131.520810   59.793307  41.153543   Apel 1.jpg   merah    merah
1  150.199144   99.803531  74.301458   Apel 7.jpg   merah    merah
2  150.162618   99.581574  74.189168   Apel 8.jpg   merah    merah
3  149.955500   99.240089  73.957600   Apel 9.jpg   merah    merah
4  149.809150   98.681830  73.623007  Apel 10.jpg   merah    merah
5   98.675641   92.972176  47.798660   Apel 2.jpg  kuning   kuning
6  100.811258   95.318118  49.051802   Apel 3.jpg  kuning   kuning
7  102.839374   97.572224  50.124256   Apel 4.jpg  kuning   kuning
8  138.429812  126.915051  73.832562   Apel 5.jpg  kuning    merah
9  137.663933  125.556890  72.544443   Apel 6.jpg  kuning    merah
